# 05 - Model Evaluation

Evaluate all models on the untouched test set.

**All metrics are calculated from actual model predictions.**

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from src.data.load_data import load_and_merge
from src.data.preprocess import run_full_preprocessing_pipeline
from src.features.feature_engineering import create_features

In [ ]:
# Load and preprocess data
df = load_and_merge('../data/raw')
df = create_features(df)

X_train, X_test, y_train, y_test, pipeline, _, _ = run_full_preprocessing_pipeline(df)

# Load trained models
stack_model = joblib.load('../models/stacking_model.pkl')
rf_model = joblib.load('../models/random_forest_model.pkl')
xgb_model = joblib.load('../models/xgboost_model.pkl')
cat_model = joblib.load('../models/catboost_model.pkl')

## Model Comparison

In [ ]:
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'CatBoost': cat_model,
    'Stacking Ensemble': stack_model,
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv('../outputs/tables/model_results.csv', index=False)

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, model) in zip(axes.flat, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'])
    ax.set_title(f'{name}')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Model Comparison Bar Chart

In [ ]:
df_melted = df_results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='Metric', y='Score', hue='Model', palette='viridis')
plt.title('Model Comparison: All Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score (0-1)', fontsize=12)
plt.ylim(0.5, 1.0)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()